# 00 Data Preparation

This notebook prepares the breast cancer DepMap/CCLE + PRISM doxorubicin modelling table.

The goal is to join baseline molecular features from DepMap/CCLE with measured PRISM doxorubicin response for the same breast cancer cell lines.

## Why This Table Is Needed

DepMap/CCLE provides baseline molecular measurements for untreated cancer cell lines. In this notebook we focus on gene expression for a small p53 and DNA-damage response gene set.

PRISM provides drug-response measurements for cancer cell lines. Here we use PRISM doxorubicin log-fold-change as the response outcome.

The joined table will later support:

- **Q2**: testing whether p53 or ODE-derived features predict doxorubicin response in breast cancer cell lines;
- **Q3**: deriving a cell-line doxorubicin-response signature and applying it to TCGA-BRCA patients;
- **Q5**: comparing mechanistic p53 features with regression or machine-learning approaches.

## Important Conceptual Point

PRISM log-fold-change is **not gene-expression change**.

In this workflow:

- DepMap/CCLE expression is the predictor matrix `X`.
- PRISM doxorubicin log-fold-change is the outcome `y`.

More negative PRISM log-fold-change generally means lower relative cell abundance or viability after doxorubicin treatment compared with control. That is interpreted as greater sensitivity.

The PRISM LFC should not be described as "percentage of cells destroyed". It is better described as relative cell abundance or viability versus control.

## Expected Raw Files

Place the DepMap/CCLE and PRISM files here:

```text
data/raw/depmap_data/
```

Expected files:

- `Model.csv`
- `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv`
- `primary-screen-replicate-collapsed-treatment-info.csv`
- `primary-screen-replicate-collapsed-logfold-change.csv`

If these files are missing, this notebook stops after the availability check and does not fabricate a processed table.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


raw_dir = Path("data/raw/depmap_data")
processed_dir = Path("data/processed")
figures_dir = Path("figures")
tables_dir = Path("tables")

required_files = {
    "model_metadata": raw_dir / "Model.csv",
    "expression": raw_dir / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    "prism_treatment_info": raw_dir / "primary-screen-replicate-collapsed-treatment-info.csv",
    "prism_lfc": raw_dir / "primary-screen-replicate-collapsed-logfold-change.csv",
}

file_check = pd.DataFrame(
    {
        "input": required_files.keys(),
        "path": [str(path) for path in required_files.values()],
        "available": [path.exists() for path in required_files.values()],
    }
)

missing_files = file_check.loc[~file_check["available"], "path"].tolist()
raw_files_available = len(missing_files) == 0

display(file_check)

if missing_files:
    print("Missing raw files. Place these files under data/raw/depmap_data/:")
    for path in missing_files:
        print(f"- {path}")

## Load DepMap/CCLE Model Metadata

First we inspect `Model.csv`, identify useful columns, and filter to breast cancer cell lines.

The most useful fields are expected to include a DepMap model ID, a cell-line name, lineage, and cancer type or primary disease.

In [ ]:
if raw_files_available:
    model_metadata = pd.read_csv(required_files["model_metadata"])

    print("Model.csv shape:", model_metadata.shape)
    print("Model.csv columns:")
    print(list(model_metadata.columns))

    id_col = "ModelID"
    name_col = "StrippedCellLineName" if "StrippedCellLineName" in model_metadata.columns else "ModelID"
    lineage_col = "OncotreeLineage" if "OncotreeLineage" in model_metadata.columns else None
    cancer_type_col = "OncotreePrimaryDisease" if "OncotreePrimaryDisease" in model_metadata.columns else None

    print("Selected ID column:", id_col)
    print("Selected cell-line name column:", name_col)
    print("Selected lineage column:", lineage_col)
    print("Selected cancer-type column:", cancer_type_col)

    lineage_text = model_metadata[lineage_col].fillna("") if lineage_col else pd.Series("", index=model_metadata.index)
    cancer_type_text = model_metadata[cancer_type_col].fillna("") if cancer_type_col else pd.Series("", index=model_metadata.index)

    breast_mask = lineage_text.str.contains("breast", case=False, na=False) | cancer_type_text.str.contains("breast", case=False, na=False)
    breast_models = model_metadata.loc[breast_mask].copy()

    breast_model_cols = [id_col, name_col]
    if lineage_col:
        breast_model_cols.append(lineage_col)
    if cancer_type_col:
        breast_model_cols.append(cancer_type_col)

    breast_models = breast_models[breast_model_cols].drop_duplicates()
    breast_models = breast_models.rename(columns={id_col: "depmap_id", name_col: "cell_line_name"})

    display(breast_models.head())
else:
    print("Skipping metadata loading until the raw files are available.")

## Load DepMap/CCLE Expression

This section keeps a small p53 and DNA-damage response gene set if those genes are present in the expression matrix.

The target gene set is:

`ATM`, `CHEK2`, `HIPK2`, `MDM2`, `PPM1D`, `SIAH1`, `TP53`, `WSB1`, `CDKN1A`, `BAX`, `BBC3`, `GADD45A`, `MDM4`, `ATR`, `CHEK1`, `CASP3`.

In [ ]:
p53_ddr_genes = [
    "ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1",
    "CDKN1A", "BAX", "BBC3", "GADD45A", "MDM4", "ATR", "CHEK1", "CASP3",
]

if raw_files_available:
    expression = pd.read_csv(required_files["expression"])
    print("Expression matrix shape:", expression.shape)
    print("First expression columns:")
    print(list(expression.columns[:10]))

    expression_id_col = "ModelID" if "ModelID" in expression.columns else expression.columns[0]
    print("Selected expression ID column:", expression_id_col)

    gene_name_map = {}
    for column in expression.columns:
        clean_gene = column.split(" (")[0]
        if clean_gene in p53_ddr_genes and clean_gene not in gene_name_map:
            gene_name_map[clean_gene] = column

    found_genes = [gene for gene in p53_ddr_genes if gene in gene_name_map]
    missing_genes = [gene for gene in p53_ddr_genes if gene not in gene_name_map]

    print("Found genes:", found_genes)
    print("Missing genes:", missing_genes)

    expression_keep_cols = [expression_id_col] + [gene_name_map[gene] for gene in found_genes]
    expression_small = expression[expression_keep_cols].copy()
    expression_small = expression_small.rename(columns={expression_id_col: "depmap_id"})
    expression_small = expression_small.rename(columns={gene_name_map[gene]: gene for gene in found_genes})

    display(expression_small.head())
else:
    print("Skipping expression loading until the raw files are available.")

## Load PRISM Treatment Information

This section searches the PRISM treatment metadata for doxorubicin. If multiple doxorubicin treatment IDs are present, we keep all of them initially and aggregate by median response per cell line later.

In [ ]:
if raw_files_available:
    treatment_info = pd.read_csv(required_files["prism_treatment_info"])

    print("Treatment info shape:", treatment_info.shape)
    print("Treatment info columns:")
    print(list(treatment_info.columns))

    treatment_name_col = "name" if "name" in treatment_info.columns else treatment_info.columns[0]
    treatment_id_col = "column_name" if "column_name" in treatment_info.columns else treatment_info.columns[0]

    doxorubicin_rows = treatment_info[
        treatment_info[treatment_name_col].astype(str).str.contains("doxorubicin", case=False, na=False)
    ].copy()

    doxorubicin_treatment_ids = doxorubicin_rows[treatment_id_col].dropna().astype(str).tolist()

    print("Selected treatment name column:", treatment_name_col)
    print("Selected treatment ID column:", treatment_id_col)
    print("Number of matched doxorubicin treatments:", len(doxorubicin_treatment_ids))
    display(doxorubicin_rows)
else:
    print("Skipping PRISM treatment-info loading until the raw files are available.")

## Load PRISM Doxorubicin Response

This section extracts the PRISM doxorubicin log-fold-change values and aggregates multiple doxorubicin treatment columns by median response per cell line.

The original PRISM doxorubicin response is kept as `prism_doxorubicin_lfc`.

In [ ]:
if raw_files_available:
    prism_lfc = pd.read_csv(required_files["prism_lfc"])

    print("PRISM LFC matrix shape:", prism_lfc.shape)
    print("First PRISM LFC columns:")
    print(list(prism_lfc.columns[:10]))

    prism_id_col = "ModelID" if "ModelID" in prism_lfc.columns else prism_lfc.columns[0]
    available_dox_cols = [col for col in doxorubicin_treatment_ids if col in prism_lfc.columns]

    print("Selected PRISM cell-line ID column:", prism_id_col)
    print("Doxorubicin response columns found in LFC matrix:", available_dox_cols)

    if available_dox_cols:
        prism_dox = prism_lfc[[prism_id_col] + available_dox_cols].copy()
        prism_dox["prism_doxorubicin_lfc"] = prism_dox[available_dox_cols].median(axis=1, skipna=True)
        prism_dox = prism_dox.rename(columns={prism_id_col: "depmap_id"})
        prism_dox = prism_dox[["depmap_id", "prism_doxorubicin_lfc"]].dropna()
        display(prism_dox.head())
    else:
        prism_dox = pd.DataFrame(columns=["depmap_id", "prism_doxorubicin_lfc"])
        print("No doxorubicin response columns were found in the PRISM LFC matrix.")
else:
    print("Skipping PRISM response loading until the raw files are available.")

## Join Breast Cancer Metadata, Expression, and PRISM Response

The modelling table joins:

```text
baseline DepMap/CCLE expression -> PRISM doxorubicin response
```

This is the supervised learning table for later modelling.

In [ ]:
if raw_files_available:
    breast_with_expression = breast_models.merge(expression_small, on="depmap_id", how="inner")
    breast_with_prism = breast_models.merge(prism_dox, on="depmap_id", how="inner")

    modelling_table = breast_with_expression.merge(prism_dox, on="depmap_id", how="inner")

    modelling_table["relative_abundance_vs_control"] = 2 ** modelling_table["prism_doxorubicin_lfc"]
    modelling_table["relative_abundance_percent"] = 100 * modelling_table["relative_abundance_vs_control"]
    modelling_table["doxorubicin_sensitivity_score"] = -1 * modelling_table["prism_doxorubicin_lfc"]

    metadata_cols = ["depmap_id", "cell_line_name"]
    if lineage_col:
        metadata_cols.append(lineage_col)
    if cancer_type_col:
        metadata_cols.append(cancer_type_col)

    response_cols = [
        "prism_doxorubicin_lfc",
        "relative_abundance_vs_control",
        "relative_abundance_percent",
        "doxorubicin_sensitivity_score",
    ]
    gene_cols = [gene for gene in found_genes if gene in modelling_table.columns]

    modelling_table = modelling_table[metadata_cols + response_cols + gene_cols].copy()

    display(modelling_table.head())
else:
    print("Skipping the final join until the raw files are available.")

## Dataset Summary and Outputs

When the raw files are available, this section saves:

- `data/processed/brca_prism_doxorubicin_modelling_table.csv`
- `tables/brca_prism_dataset_summary.csv`
- `figures/prism_doxorubicin_lfc_distribution.png`

In [ ]:
if raw_files_available:
    processed_dir.mkdir(exist_ok=True)
    tables_dir.mkdir(exist_ok=True)
    figures_dir.mkdir(exist_ok=True)

    summary = pd.DataFrame(
        {
            "metric": [
                "total DepMap models",
                "breast cancer models",
                "breast cancer models with expression data",
                "breast cancer models with PRISM doxorubicin response",
                "final joined modelling table rows",
            ],
            "value": [
                len(model_metadata),
                len(breast_models),
                len(breast_with_expression),
                len(breast_with_prism),
                len(modelling_table),
            ],
        }
    )

    modelling_table_path = processed_dir / "brca_prism_doxorubicin_modelling_table.csv"
    summary_path = tables_dir / "brca_prism_dataset_summary.csv"
    figure_path = figures_dir / "prism_doxorubicin_lfc_distribution.png"

    modelling_table.to_csv(modelling_table_path, index=False)
    summary.to_csv(summary_path, index=False)

    plt.figure(figsize=(7, 4.5))
    plt.hist(modelling_table["prism_doxorubicin_lfc"].dropna(), bins=15, edgecolor="black")
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.xlabel("PRISM doxorubicin log-fold-change")
    plt.ylabel("Number of breast cancer cell lines")
    plt.title("Distribution of PRISM doxorubicin response")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.show()

    display(summary)
    print("Saved modelling table to:", modelling_table_path)
    print("Saved summary table to:", summary_path)
    print("Saved figure to:", figure_path)
else:
    print("No processed table, summary CSV, or figure was created because raw files are missing.")

## Interpretation

When the raw files are available and the notebook is run, report:

- how many DepMap models were available;
- how many were breast cancer cell lines;
- how many breast cancer cell lines had expression data;
- how many breast cancer cell lines had PRISM doxorubicin response;
- how many rows are in the final joined modelling table.

A negative PRISM doxorubicin LFC means lower relative abundance or viability compared with control after doxorubicin treatment. In this notebook, `doxorubicin_sensitivity_score = -1 * prism_doxorubicin_lfc`, so higher sensitivity scores mean greater apparent sensitivity.

This table is the basis for later regression and signature modelling because it links baseline cell-line expression features to measured doxorubicin response. No lasso, elastic net, survival analysis, GDSC analysis, LINCS analysis, or p53 ODE implementation is run here.